## Data Preparation

### 1. Persiapan

#### 1.1. Import Library

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from config import (
  RAW_PROVINCES_CSV,
  RAW_REGENCIES_CSV,
  GEO_PROVINCES_JSON,
  GEO_REGENCIES_JSON,
  CLEANED_PROVINCES_CSV,
  CLEANED_REGENCIES_CSV,
  SCALED_FEATURES_CSV,
  NUMERIC_COLUMNS,
  FEATURE_COLUMNS
)

#### 1.2. Persiapan Data

In [ ]:
df_prov = pd.read_csv(RAW_PROVINCES_CSV)
df_reg = pd.read_csv(RAW_REGENCIES_CSV)

### 2. Pembersihan Data & Penggabungan Geospasial

#### 2.1. Pembersihan & Kalkulasi Rasio Tingkat Provinsi

In [ ]:
df_prov['province_name'] = df_prov['province_name'].astype(str).str.strip().str.upper()

if os.path.exists(GEO_PROVINCES_JSON):
  with open(GEO_PROVINCES_JSON, encoding='utf-8') as f:
    geo_p = pd.DataFrame(json.load(f))
  if not geo_p.empty:
    geo_p['province_name_clean'] = geo_p['name'].astype(str).str.strip().str.upper()
    df_prov = df_prov.merge(
      geo_p[['province_name_clean', 'province_id', 'latitude', 'longitude']],
      left_on='province_name',
      right_on='province_name_clean',
      how='left'
    ).drop(columns=['province_name_clean'], errors='ignore')

df_prov['rasio_nib'] = np.where(df_prov['total_koperasi'] > 0, (df_prov['koperasi_nib'] / df_prov['total_koperasi']) * 100.0, 0.0).clip(0, 100).round(2)
df_prov['rasio_npwp'] = np.where(df_prov['total_koperasi'] > 0, (df_prov['koperasi_npwp'] / df_prov['total_koperasi']) * 100.0, 0.0).clip(0, 100).round(2)
df_prov['rasio_rat'] = np.where(df_prov['total_koperasi'] > 0, (df_prov['koperasi_rat'] / df_prov['total_koperasi']) * 100.0, 0.0).clip(0, 100).round(2)

os.makedirs(os.path.dirname(CLEANED_PROVINCES_CSV), exist_ok=True)
df_prov.to_csv(CLEANED_PROVINCES_CSV, index=False)

df_prov.head()

#### 2.2. Imputasi & Penggabungan Geospasial Tingkat Kabupaten/Kota

In [ ]:
df_reg['regency_name'] = df_reg['regency_name'].astype(str).str.strip().str.upper()

for col in NUMERIC_COLUMNS:
  if col in df_reg.columns:
    df_reg[col] = df_reg.groupby('province_id')[col].transform(
      lambda s: s.fillna(s.median() if not s.dropna().empty else 0)
    ).fillna(0)

if os.path.exists(GEO_REGENCIES_JSON):
  with open(GEO_REGENCIES_JSON, encoding='utf-8') as f:
    geo_r = pd.DataFrame(json.load(f))
  if not geo_r.empty:
    df_reg = df_reg.merge(
      geo_r[['province_id', 'regency_no', 'latitude', 'longitude']],
      on=['province_id', 'regency_no'],
      how='left'
    )

df_reg['rasio_nib'] = np.where(df_reg['total_koperasi'] > 0, (df_reg['koperasi_nib'] / df_reg['total_koperasi']) * 100.0, 0.0).clip(0, 100).round(2)
df_reg['rasio_npwp'] = np.where(df_reg['total_koperasi'] > 0, (df_reg['koperasi_npwp'] / df_reg['total_koperasi']) * 100.0, 0.0).clip(0, 100).round(2)
df_reg['rasio_rat'] = np.where(df_reg['total_koperasi'] > 0, (df_reg['koperasi_rat'] / df_reg['total_koperasi']) * 100.0, 0.0).clip(0, 100).round(2)

os.makedirs(os.path.dirname(CLEANED_REGENCIES_CSV), exist_ok=True)
df_reg.to_csv(CLEANED_REGENCIES_CSV, index=False)

print(f"Total Baris Provinsi Bersih        : {len(df_prov)}")
print(f"Total Baris Kabupaten/Kota Bersih : {len(df_reg)}")

### 3. Transformasi & Standardisasi Fitur

#### 3.1. Transformasi Logaritmik (Log1p)

In [ ]:
LOG_TRANSFORM_FEATURES = [
  'total_koperasi',
  'simpanan_pokok',
  'simpanan_wajib',
  'volume_transaksi',
  'nilai_transaksi'
]

features_present = [col for col in FEATURE_COLUMNS if col in df_reg.columns]
X_df = df_reg[features_present].copy().fillna(0)

for col in LOG_TRANSFORM_FEATURES:
  if col in X_df.columns:
    X_df[col] = np.log1p(np.maximum(X_df[col].values, 0))

X_df.describe().round(2).T

#### 3.2. Standardisasi Fitur (Z-Score Scaling)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_df.values)

scaled_df = pd.DataFrame(X_scaled, columns=[f"scaled_{c}" for c in features_present])
os.makedirs(os.path.dirname(SCALED_FEATURES_CSV), exist_ok=True)
scaled_df.to_csv(SCALED_FEATURES_CSV, index=False)

print(f"Total Fitur Terstandarisasi : {len(features_present)}")
print(f"Bentuk Matriks Fitur        : {scaled_df.shape}")
scaled_df.head()